# 91 Colab Unified Unsloth First

Status: Canonical Colab surface | Draft until a real Colab G4 run produces `workspace/runs/<run_id>/`.

Guarantees:
1. self-contained runtime bootstrap from embedded helper code and embedded config snapshots
2. Unsloth-first install path by default, with a matrix-based Unsloth fallback only after the primary path fails
3. safe Drive mount recovery
4. automatic identity bootstrap from `STnoui/lumis1-identity`
5. multimodal SFT as the main training path, not a hidden text-only fallback
6. GGUF export first from the strongest completed artifact of the current run
7. automatic browser download of the final artifact or `final_deliverables.zip`

Non-guarantees:
1. multimodal DPO is not claimed as stable on this path; the default policy skips it on a multimodal run when only text preferences are available
2. proof-bearing success still requires a real Colab G4 run tree under `workspace/runs/`


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK_SELF_CONTAINED = True
WORK_ROOT = Path("/content/lumis1_unified")
WORKSPACE_ROOT = WORK_ROOT / "workspace"
REPORTS_ROOT = WORKSPACE_ROOT / "reports"
FINAL_ROOT = WORKSPACE_ROOT / "final"
RUNS_ROOT = WORKSPACE_ROOT / "runs"
ARTIFACTS_ROOT = WORKSPACE_ROOT / "artifacts"
CHECKSUMS_ROOT = WORKSPACE_ROOT / "checksums"
REPO_ROOT = Path("/content/lumis1_unified_repo_stub")
DRIVE_ROOT = Path("/content/drive/MyDrive/lumis1_unified")
IDENTITY_INPUT_DIR = WORK_ROOT / "identity_input"
IDENTITY_REPO_ID = os.environ.get("LUMIS1_IDENTITY_HF_REPO", "STnoui/lumis1-identity")
INSTALL_STRATEGY = "unsloth_first"
PROFILE = "colab_g4_first_run"
BASE_MODEL = "Qwen/Qwen3.5-4B"
EXPORT_QUANTIZATION_METHODS = ["q4_k_m", "q8_0"]
EXPERIMENTAL_DPO = False
RUN_ID = datetime.now(timezone.utc).strftime("colab91-%Y%m%dT%H%M%SZ")
RUN_ROOT = RUNS_ROOT / RUN_ID
STATUS_PATH = RUN_ROOT / "STATUS.json"
SUMMARY_PATH = RUN_ROOT / "SUMMARY.md"
BOOTSTRAP_ROOT = RUN_ROOT / "bootstrap"
DATASET_RUN_ROOT = RUN_ROOT / "dataset"
SFT_RUN_ROOT = RUN_ROOT / "sft"
EXPORT_SFT_RUN_ROOT = RUN_ROOT / "export_sft"
DPO_RUN_ROOT = RUN_ROOT / "dpo"
EXPORT_FINAL_RUN_ROOT = RUN_ROOT / "export_final"
EVAL_RUN_ROOT = RUN_ROOT / "eval"
RUN_STAGE_DIRS = {
    "bootstrap": BOOTSTRAP_ROOT,
    "dataset": DATASET_RUN_ROOT,
    "sft": SFT_RUN_ROOT,
    "export_sft": EXPORT_SFT_RUN_ROOT,
    "dpo": DPO_RUN_ROOT,
    "export_final": EXPORT_FINAL_RUN_ROOT,
    "eval": EVAL_RUN_ROOT,
    "artifacts": RUN_ROOT / "artifacts",
    "checksums": RUN_ROOT / "checksums",
}
EMBEDDED_CONFIG_TEXT = {"dataset_mixture.yaml": "version: \"2.1\"\nproject: \"Lumis-1\"\n\nidentity_pack:\n  required_files:\n    sft_candidates:\n      - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/sft_dataset.jsonl\"\n      - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/identity_sft.jsonl\"\n    preferences_candidates:\n      - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/preference_dataset.jsonl\"\n      - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/identity_preferences.jsonl\"\n    report_pdf_optional_candidates:\n      - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/identity_pack_report.pdf\"\n  required_counts:\n    sft_rows: 100000\n    preference_rows: 25000\n  fixed_share_of_final_sft_tokens: 0.20\n  enforce_exact_token_share: true\n\ntoken_budgets:\n  global_sft_tokens:\n    mode: \"derive_from_identity_exact_share\"\n    explicit_value: null\n  open_sft_budget:\n    mode: \"derive_from_identity_exact_share\"\n    dry_run_tokens: 50000\n  per_source_default_tokens: 20000000\n  preferences_global_token_budget: 120000000\n\ntargets:\n  category_share:\n    polished_general_assistant: 0.30\n    real_user_conversations: 0.20\n    multilingual: 0.15\n    utility_tasks: 0.15\n    identity_behavior: 0.20\n  modality_share:\n    text: 0.88\n    image_text: 0.12\n  tolerance:\n    row_share_abs: 0.0100\n    token_share_abs: 0.0150\n\nnon_identity_multimodal_derivation:\n  formula: \"(overall_mm - identity_share * identity_mm) / (1 - identity_share)\"\n  inputs:\n    overall_mm: 0.12\n    identity_share: 0.20\n    identity_mm_source: \"computed_from_notebook_10\"\n  enforce: true\n\ningestion_defaults:\n  source_mode: \"hf\"\n  streaming_preferred: true\n  max_records_scanned_per_source: 2000000\n  enforce_allowlist: true\n\nsources:\n  - source_id: \"HuggingFaceH4/ultrachat_200k\"\n    category: \"polished_general_assistant\"\n    modality: \"text\"\n    token_budget: 130000000\n  - source_id: \"CohereLabs/aya_dataset\"\n    category: \"multilingual\"\n    modality: \"text\"\n    token_budget: 90000000\n  - source_id: \"allenai/WildChat-4.8M\"\n    category: \"real_user_conversations\"\n    modality: \"text\"\n    token_budget: 120000000\n  - source_id: \"HuggingFaceTB/smoltalk2\"\n    category: \"utility_tasks\"\n    modality: \"text\"\n    token_budget: 75000000\n    subset_rule: \"no_think_only\"\n  - source_id: \"nvidia/HelpSteer3\"\n    category: \"utility_tasks\"\n    modality: \"text\"\n    token_budget: 35000000\n    preference_candidate: true\n  - source_id: \"argilla/ultrafeedback-binarized-preferences-cleaned\"\n    category: \"utility_tasks\"\n    modality: \"text\"\n    token_budget: 40000000\n    preference_candidate: true\n  - source_id: \"HuggingFaceM4/the_cauldron\"\n    category: \"utility_tasks\"\n    modality: \"image_text\"\n    token_budget: 16000000\n  - source_id: \"HuggingFaceM4/Docmatix\"\n    category: \"utility_tasks\"\n    modality: \"image_text\"\n    token_budget: 14000000\n  - source_id: \"facebook/textvqa\"\n    category: \"utility_tasks\"\n    modality: \"image_text\"\n    token_budget: 10000000\n  - source_id: \"lmms-lab/DocVQA\"\n    category: \"utility_tasks\"\n    modality: \"image_text\"\n    token_budget: 10000000\n\npreferences:\n  required_output: \"workspace/final/full_preferences.jsonl\"\n  mix_targets:\n    identity_preferences: 0.25\n    ultrafeedback_cleaned: 0.60\n    helpsteer3: 0.15\n  enabled_sources:\n    - \"argilla/ultrafeedback-binarized-preferences-cleaned\"\n    - \"nvidia/HelpSteer3\"\n  optional_additional_preferences:\n    enabled: false\n    sources: []\n\noutput_paths:\n  open_sft_interim: \"workspace/interim/open_sft.jsonl\"\n  open_preferences_interim: \"workspace/interim/open_preferences.jsonl\"\n  final_sft: \"workspace/final/full_sft.jsonl\"\n  final_preferences: \"workspace/final/full_preferences.jsonl\"\n  manifest: \"workspace/final/dataset_manifest.json\"\n  reports:\n    open_corpus: \"workspace/reports/open_corpus_build_report.json\"\n    identity_validation: \"workspace/reports/identity_validation.json\"\n    full_validation: \"workspace/reports/full_dataset_validation.json\"\n    mixture_math: \"workspace/reports/mixture_math.json\"\n", "dataset_sources_allowlist.yaml": "version: \"2.1\"\npolicy:\n  default_action: \"deny\"\n  allow_only_listed_sources: true\n  reject_unlisted_sources: true\n  private_local_requires_attestation: true\n\nsources:\n  - source_id: \"HuggingFaceH4/ultrachat_200k\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"MIT\"\n    category: [\"polished_general_assistant\"]\n    default_split: \"train_sft\"\n    max_tokens_cap: 180000000\n    link: \"https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k\"\n\n  - source_id: \"CohereLabs/aya_dataset\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"Apache-2.0\"\n    category: [\"multilingual\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 140000000\n    link: \"https://huggingface.co/datasets/CohereLabs/aya_dataset\"\n\n  - source_id: \"allenai/WildChat-4.8M\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"ODC-BY\"\n    category: [\"real_user_conversations\", \"multilingual\"]\n    default_split: \"train\"\n    max_tokens_cap: 220000000\n    strip_fields:\n      - \"age\"\n      - \"gender\"\n      - \"race\"\n      - \"ethnicity\"\n      - \"demographics\"\n    link: \"https://huggingface.co/datasets/allenai/WildChat-4.8M\"\n\n  - source_id: \"HuggingFaceTB/smoltalk2\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"subset_specific\"\n    category: [\"polished_general_assistant\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 130000000\n    require_thinking_off: true\n    subset_allowlist:\n      mode: \"allow_only\"\n      rules:\n        - subset_name_pattern: \"*no_think*\"\n          required_license_in:\n            - \"Apache-2.0\"\n            - \"MIT\"\n            - \"CC-BY-4.0\"\n    link: \"https://huggingface.co/datasets/HuggingFaceTB/smoltalk2\"\n\n  - source_id: \"nvidia/HelpSteer3\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"CC-BY-4.0\"\n    category: [\"utility_tasks\", \"preference_dpo\"]\n    default_split: \"train\"\n    max_tokens_cap: 80000000\n    dpo_secondary: true\n    link: \"https://huggingface.co/datasets/nvidia/HelpSteer3\"\n\n  - source_id: \"argilla/ultrafeedback-binarized-preferences-cleaned\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"MIT\"\n    category: [\"preference_dpo\"]\n    default_split: \"train\"\n    max_tokens_cap: 200000000\n    dpo_primary: true\n    link: \"https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned\"\n\n  - source_id: \"HuggingFaceM4/the_cauldron\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"subset_specific\"\n    category: [\"image_text\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 30000000\n    sampled_only: true\n    subset_allowlist:\n      mode: \"allow_only\"\n      required_subset_license_in:\n        - \"MIT\"\n        - \"Apache-2.0\"\n        - \"CC-BY-4.0\"\n      subsets: []\n    prompts_license_note: \"Prompts are CC-BY-4.0; each subset still requires explicit subset license approval.\"\n    link: \"https://huggingface.co/datasets/HuggingFaceM4/the_cauldron\"\n\n  - source_id: \"HuggingFaceM4/Docmatix\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"MIT\"\n    category: [\"image_text\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 24000000\n    sampled_only: true\n    link: \"https://huggingface.co/datasets/HuggingFaceM4/Docmatix\"\n\n  - source_id: \"facebook/textvqa\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"CC-BY-4.0\"\n    category: [\"image_text\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 16000000\n    sampled_only: true\n    link: \"https://huggingface.co/datasets/facebook/textvqa\"\n\n  - source_id: \"lmms-lab/DocVQA\"\n    source_mode: \"hf\"\n    enabled: true\n    license: \"Apache-2.0\"\n    category: [\"image_text\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 16000000\n    sampled_only: true\n    link: \"https://huggingface.co/datasets/lmms-lab/DocVQA\"\n\n  - source_id: \"HuggingFaceM4/FineVision\"\n    source_mode: \"hf\"\n    enabled: false\n    license: \"subset_specific\"\n    category: [\"image_text\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 20000000\n    sampled_only: true\n    subset_allowlist:\n      mode: \"allow_only\"\n      required_subset_license_in:\n        - \"MIT\"\n        - \"Apache-2.0\"\n        - \"CC-BY-4.0\"\n      subsets: []\n    operator_approval_required: true\n    warning: \"Disabled by default. Enable only after explicit subset-level license approval.\"\n    link: \"https://huggingface.co/datasets/HuggingFaceM4/FineVision\"\n\n  - source_id: \"nvidia/Nemotron-Post-Training-Dataset-v2\"\n    source_mode: \"hf\"\n    enabled: false\n    license: \"CC-BY-4.0\"\n    category: [\"preference_dpo\", \"utility_tasks\"]\n    default_split: \"train\"\n    max_tokens_cap: 0\n    operator_approval_required: true\n    provenance_attestation_required: true\n    warning: \"Disabled by default. Requires explicit operator provenance and licensing approval before use.\"\n    link: \"https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2\"\n\n  - source_id: \"allenai/tulu-3-sft-mixture\"\n    source_mode: \"hf\"\n    enabled: false\n    license: \"subset_specific\"\n    category: [\"optional\"]\n    default_split: \"train\"\n    max_tokens_cap: 0\n    operator_approval_required: true\n    subset_allowlist:\n      mode: \"allow_only\"\n      subsets: []\n    link: \"https://huggingface.co/datasets/allenai/tulu-3-sft-mixture\"\n\n  - source_id: \"PRIVATE_LOCAL_01\"\n    source_mode: \"local\"\n    enabled: false\n    local_path: \"\"\n    category: [\"operator_defined\"]\n    max_tokens_cap: 0\n    provenance_attestation: \"\"\n    license: \"\"\n    redistribution_allowed: false\n    pii_policy: \"\"\n\n  - source_id: \"PRIVATE_LOCAL_02\"\n    source_mode: \"local\"\n    enabled: false\n    local_path: \"\"\n    category: [\"operator_defined\"]\n    max_tokens_cap: 0\n    provenance_attestation: \"\"\n    license: \"\"\n    redistribution_allowed: false\n    pii_policy: \"\"\n", "train_sft.yaml": "﻿version: \"2.0\"\nmode: \"sft\"\n\nmodel:\n  base_model: \"Qwen/Qwen3.5-4B\"\n  vision_capable: true\n  dtype: \"bf16\"\n  load_in_4bit: false\n  full_finetuning: false\n\nlora:\n  enabled: true\n  r: 32\n  lora_alpha: 64\n  lora_dropout: 0.0\n  bias: \"none\"\n  target_modules:\n    - \"q_proj\"\n    - \"k_proj\"\n    - \"v_proj\"\n    - \"o_proj\"\n    - \"gate_proj\"\n    - \"up_proj\"\n    - \"down_proj\"\n\ntraining:\n  per_device_train_batch_size: 2\n  gradient_accumulation_steps: 16\n  learning_rate: 2.0e-5\n  warmup_steps: 50\n  max_steps: 3000\n  logging_steps: 5\n  save_steps: 200\n  lr_scheduler_type: \"cosine\"\n  optim: \"adamw_torch\"\n  weight_decay: 0.01\n  bf16: true\n\nsanity_run:\n  enabled_by_default: true\n  max_steps: 50\n\nchat_template_policy:\n  config_path: \"configs/chat_template_policy.yaml\"\n  enforce_enable_thinking_false: true\n\ndatasets:\n  train_sft_path: \"workspace/final/full_sft.jsonl\"\n  dataset_manifest_path: \"workspace/final/dataset_manifest.json\"\n\noutputs:\n  run_dir: \"workspace/runs/manual-sft/artifacts/sft_model\"\n  resolved_config_report: \"workspace/reports/train_sft_config_resolved.json\"\n  checkpoint_dir: \"workspace/runs/manual-sft/artifacts/sft_model/checkpoints\"\n\nexecution:\n  run_training_default: false\n  local_only: true\n", "train_dpo.yaml": "﻿version: \"2.0\"\nmode: \"dpo\"\n\nmodel:\n  base_model: \"Qwen/Qwen3.5-4B\"\n  sft_checkpoint_or_adapter: \"workspace/runs/manual-sft/artifacts/sft_model\"\n  vision_capable: true\n  dtype: \"bf16\"\n  load_in_4bit: false\n\nlora:\n  enabled: true\n  r: 32\n  lora_alpha: 64\n  lora_dropout: 0.0\n  bias: \"none\"\n\ntraining:\n  per_device_train_batch_size: 2\n  gradient_accumulation_steps: 16\n  learning_rate: 1.0e-5\n  warmup_steps: 50\n  max_steps: 2000\n  logging_steps: 5\n  save_steps: 200\n  lr_scheduler_type: \"cosine\"\n  optim: \"adamw_torch\"\n  bf16: true\n\ndpo:\n  beta: 0.1\n  loss_type: \"sigmoid\"\n  reference_free: false\n\npreferences:\n  required_identity_preferences_path: \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/preference_dataset.jsonl\"\n  required_identity_preferences_candidates:\n    - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/preference_dataset.jsonl\"\n    - \"Dataset/identity_dataset/output/full_run_codex_spark_xhigh/identity_preferences.jsonl\"\n  ultrafeedback_path_hint: \"argilla/ultrafeedback-binarized-preferences-cleaned\"\n  helpsteer3_path_hint: \"nvidia/HelpSteer3\"\n  mix:\n    identity_preferences: 0.25\n    ultrafeedback_cleaned: 0.60\n    helpsteer3: 0.15\n  optional_additional_preferences:\n    enabled: false\n    sources: []\n\nchat_template_policy:\n  config_path: \"configs/chat_template_policy.yaml\"\n  enforce_enable_thinking_false: true\n\noutputs:\n  run_dir: \"workspace/runs/manual-dpo/artifacts/dpo_model\"\n  resolved_config_report: \"workspace/reports/train_dpo_config_resolved.json\"\n\nexecution:\n  run_training_default: false\n  local_only: true\n", "run_profiles.yaml": "﻿version: \"2.0\"\nprofiles:\n  colab_g4_first_run:\n    description: \"Primary Colab G4 profile for notebook 91 first-run multimodal export\"\n    sft:\n      per_device_train_batch_size: 1\n      gradient_accumulation_steps: 16\n      max_seq_length: 3072\n    dpo:\n      per_device_train_batch_size: 1\n      gradient_accumulation_steps: 16\n      max_seq_length: 3072\n\n  colab_g4_first_run:\n    description: \"Default first serious Colab G4 profile for notebook 91\"\n    sft:\n      per_device_train_batch_size: 1\n      gradient_accumulation_steps: 16\n      max_seq_length: 3072\n    dpo:\n      per_device_train_batch_size: 1\n      gradient_accumulation_steps: 16\n      max_seq_length: 3072\n\n  default_96gb:\n    description: \"Primary profile for 96GB-class GPU memory budget\"\n    sft:\n      per_device_train_batch_size: 2\n      gradient_accumulation_steps: 16\n      max_seq_length: 4096\n    dpo:\n      per_device_train_batch_size: 2\n      gradient_accumulation_steps: 16\n      max_seq_length: 4096\n\n  safe_fallback:\n    description: \"Reduced memory profile for constrained GPUs\"\n    sft:\n      per_device_train_batch_size: 1\n      gradient_accumulation_steps: 32\n      max_seq_length: 3072\n    dpo:\n      per_device_train_batch_size: 1\n      gradient_accumulation_steps: 32\n      max_seq_length: 3072\n"}
EMBEDDED_RUNTIME_SOURCE = "\"\"\"Repo-tested helpers embedded into notebook 91 at runtime.\n\nThis module must remain self-contained enough that its source can be embedded\ndirectly into a Colab notebook without importing repo-side modules there.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport hashlib\nimport io\nimport json\nimport textwrap\nimport urllib.request\nimport zipfile\nfrom collections import Counter\nfrom collections.abc import Iterable\nfrom pathlib import Path\nfrom typing import Any\n\nfrom packaging.version import Version\nfrom PIL import Image, ImageDraw\n\n\nPLACEHOLDER_PREFIXES = (\"image://\", \"synthetic://\")\nTEXT_ONLY_SOURCES = {\n    \"HuggingFaceH4/ultrachat_200k\",\n    \"CohereLabs/aya_dataset\",\n    \"allenai/WildChat-4.8M\",\n    \"HuggingFaceTB/smoltalk2\",\n    \"nvidia/HelpSteer3\",\n    \"argilla/ultrafeedback-binarized-preferences-cleaned\",\n}\nMULTIMODAL_SOURCES = {\n    \"HuggingFaceM4/Docmatix\",\n    \"facebook/textvqa\",\n    \"lmms-lab/DocVQA\",\n    \"HuggingFaceM4/the_cauldron\",\n}\nCORE_STACK_PACKAGES = {\n    \"unsloth\",\n    \"unsloth_zoo\",\n    \"torch\",\n    \"transformers\",\n    \"trl\",\n    \"accelerate\",\n    \"peft\",\n    \"bitsandbytes\",\n    \"torchvision\",\n}\nIDENTITY_ALLOW_PATTERNS = [\n    \"sft_dataset.jsonl\",\n    \"preference_dataset.jsonl\",\n    \"identity_pack_report.pdf\",\n]\n\n\ndef sha256_file(path: str | Path) -> str:\n    target = Path(path)\n    digest = hashlib.sha256()\n    with target.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef normalize_text(value: Any) -> str:\n    if isinstance(value, str):\n        return \" \".join(value.replace(\"\\u00a0\", \" \").split()).strip()\n    if isinstance(value, list):\n        parts = [normalize_text(item) for item in value]\n        return \" \".join(part for part in parts if part).strip()\n    if isinstance(value, dict):\n        if isinstance(value.get(\"text\"), str):\n            return normalize_text(value[\"text\"])\n        if isinstance(value.get(\"content\"), str):\n            return normalize_text(value[\"content\"])\n    return \"\"\n\n\ndef best_effort_user_text(messages: list[dict[str, Any]]) -> str:\n    parts: list[str] = []\n    for message in messages:\n        if not isinstance(message, dict) or message.get(\"role\") != \"user\":\n            continue\n        content = message.get(\"content\")\n        if isinstance(content, str):\n            text = normalize_text(content)\n            if text:\n                parts.append(text)\n            continue\n        if isinstance(content, list):\n            for block in content:\n                if isinstance(block, dict) and block.get(\"type\") == \"text\":\n                    text = normalize_text(block.get(\"text\"))\n                    if text:\n                        parts.append(text)\n    return \"\\n\".join(parts).strip()\n\n\ndef _ensure_parent(path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n\ndef create_surrogate_document_image(prompt_text: str, destination: str | Path) -> Path:\n    dest = Path(destination).expanduser().resolve()\n    _ensure_parent(dest)\n    image = Image.new(\"RGB\", (1344, 1024), (248, 248, 245))\n    draw = ImageDraw.Draw(image)\n    draw.rectangle((48, 48, 1296, 976), outline=(188, 188, 188), width=3)\n    draw.rectangle((84, 84, 1260, 160), fill=(225, 229, 235))\n    draw.text((108, 112), \"Lumis-1 surrogate document context\", fill=(32, 32, 32))\n    wrapped = textwrap.wrap(prompt_text or \"Image context placeholder\", width=58)[:22]\n    y = 210\n    for line in wrapped:\n        draw.text((108, y), line, fill=(24, 24, 24))\n        y += 34\n    draw.text((108, 922), \"Generated from placeholder image reference\", fill=(90, 90, 90))\n    image.save(dest, format=\"PNG\")\n    return dest\n\n\ndef materialize_image_value(image_value: Any, destination: str | Path) -> Path:\n    dest = Path(destination).expanduser().resolve()\n    _ensure_parent(dest)\n\n    if isinstance(image_value, Image.Image):\n        image_value.save(dest)\n        return dest\n\n    if isinstance(image_value, dict):\n        if isinstance(image_value.get(\"bytes\"), (bytes, bytearray)):\n            Image.open(io.BytesIO(image_value[\"bytes\"])).save(dest)\n            return dest\n        if isinstance(image_value.get(\"bytes\"), str) and image_value[\"bytes\"].strip():\n            raw = base64.b64decode(image_value[\"bytes\"])\n            Image.open(io.BytesIO(raw)).save(dest)\n            return dest\n        if isinstance(image_value.get(\"path\"), str) and image_value[\"path\"].strip():\n            return materialize_image_value(image_value[\"path\"], dest)\n        if isinstance(image_value.get(\"url\"), str) and image_value[\"url\"].strip():\n            return materialize_image_value(image_value[\"url\"], dest)\n\n    if isinstance(image_value, (bytes, bytearray)):\n        Image.open(io.BytesIO(image_value)).save(dest)\n        return dest\n\n    if isinstance(image_value, str) and image_value.strip():\n        value = image_value.strip()\n        lowered = value.lower()\n        if lowered.startswith((\"http://\", \"https://\")):\n            with urllib.request.urlopen(value) as response:\n                raw = response.read()\n            Image.open(io.BytesIO(raw)).save(dest)\n            return dest\n        path = Path(value).expanduser()\n        if path.exists():\n            image = Image.open(path)\n            image.save(dest)\n            return dest\n\n    raise ValueError(f\"unsupported image value for materialization: {type(image_value)!r}\")\n\n\ndef make_image_block(path: str | Path) -> dict[str, str]:\n    resolved = str(Path(path).expanduser().resolve())\n    return {\n        \"type\": \"image\",\n        \"image\": resolved,\n        \"path\": resolved,\n        \"image_path\": resolved,\n    }\n\n\ndef normalize_messages_as_blocks(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    normalized: list[dict[str, Any]] = []\n    for message in messages:\n        if not isinstance(message, dict):\n            continue\n        role = str(message.get(\"role\") or \"user\")\n        content = message.get(\"content\")\n        if isinstance(content, str):\n            normalized.append({\"role\": role, \"content\": [{\"type\": \"text\", \"text\": content}]})\n            continue\n        if isinstance(content, list):\n            blocks: list[dict[str, Any]] = []\n            for block in content:\n                if not isinstance(block, dict):\n                    continue\n                if block.get(\"type\") == \"text\":\n                    text = normalize_text(block.get(\"text\"))\n                    if text:\n                        blocks.append({\"type\": \"text\", \"text\": text})\n                elif block.get(\"type\") == \"image\":\n                    image_block = {\"type\": \"image\"}\n                    for key in (\"image\", \"path\", \"image_path\", \"image_bytes_b64\"):\n                        if isinstance(block.get(key), str) and str(block[key]).strip():\n                            image_block[key] = str(block[key]).strip()\n                    blocks.append(image_block)\n            if blocks:\n                normalized.append({\"role\": role, \"content\": blocks})\n    return normalized\n\n\ndef materialize_identity_placeholder_assets(\n    rows: list[dict[str, Any]],\n    asset_root: str | Path,\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    asset_dir = Path(asset_root).expanduser().resolve()\n    asset_dir.mkdir(parents=True, exist_ok=True)\n    materialized: list[dict[str, Any]] = []\n    placeholder_rows = 0\n    created_images = 0\n\n    for row in rows:\n        current = json.loads(json.dumps(row))\n        current[\"messages\"] = normalize_messages_as_blocks(list(current.get(\"messages\") or []))\n        prompt_text = best_effort_user_text(current[\"messages\"])\n        for message in current[\"messages\"]:\n            content = message.get(\"content\")\n            if not isinstance(content, list):\n                continue\n            for block in content:\n                if not isinstance(block, dict) or block.get(\"type\") != \"image\":\n                    continue\n                image_ref = block.get(\"image\")\n                if isinstance(image_ref, str) and image_ref.startswith(PLACEHOLDER_PREFIXES):\n                    placeholder_rows += 1\n                    stem = hashlib.sha256(f\"{current.get('id')}::{image_ref}\".encode(\"utf-8\")).hexdigest()[:20]\n                    asset_path = asset_dir / f\"{stem}.png\"\n                    if not asset_path.exists():\n                        create_surrogate_document_image(prompt_text, asset_path)\n                        created_images += 1\n                    block.update(make_image_block(asset_path))\n        current[\"modality\"] = \"image_text\" if \"image_path\" in json.dumps(current, ensure_ascii=False) else \"text\"\n        materialized.append(current)\n\n    return materialized, {\n        \"rows_input\": len(rows),\n        \"rows_output\": len(materialized),\n        \"placeholder_blocks_seen\": placeholder_rows,\n        \"surrogate_images_created\": created_images,\n        \"asset_root\": str(asset_dir),\n    }\n\n\ndef _pick_first_text(*candidates: Any) -> str:\n    for candidate in candidates:\n        text = normalize_text(candidate)\n        if text:\n            return text\n    return \"\"\n\n\ndef _pick_majority_answer(values: Any) -> str:\n    if isinstance(values, str):\n        return normalize_text(values)\n    if isinstance(values, list):\n        normalized = [normalize_text(item) for item in values if normalize_text(item)]\n        if not normalized:\n            return \"\"\n        return Counter(normalized).most_common(1)[0][0]\n    return \"\"\n\n\ndef _save_named_image(source_id: str, row_id: str, image_value: Any, asset_root: Path) -> Path:\n    stem = hashlib.sha256(f\"{source_id}::{row_id}\".encode(\"utf-8\")).hexdigest()[:20]\n    destination = asset_root / source_id.replace(\"/\", \"__\") / f\"{stem}.png\"\n    if destination.exists():\n        return destination\n    return materialize_image_value(image_value, destination)\n\n\ndef build_multimodal_row_from_record(\n    source_id: str,\n    record: dict[str, Any],\n    *,\n    asset_root: str | Path,\n    row_id: str,\n) -> dict[str, Any] | None:\n    asset_dir = Path(asset_root).expanduser().resolve()\n    asset_dir.mkdir(parents=True, exist_ok=True)\n\n    if source_id == \"facebook/textvqa\":\n        question = _pick_first_text(record.get(\"question\"))\n        answer = _pick_majority_answer(record.get(\"answers\"))\n        image_value = record.get(\"image\")\n        if question and answer and image_value is not None:\n            image_path = _save_named_image(source_id, row_id, image_value, asset_dir)\n            return {\n                \"id\": row_id,\n                \"source_id\": source_id,\n                \"license\": \"CC-BY-4.0\",\n                \"thinking\": \"off\",\n                \"chat_template_kwargs\": {\"enable_thinking\": False},\n                \"category\": \"utility_tasks\",\n                \"modality\": \"image_text\",\n                \"messages\": [\n                    {\n                        \"role\": \"user\",\n                        \"content\": [\n                            make_image_block(image_path),\n                            {\"type\": \"text\", \"text\": question},\n                        ],\n                    },\n                    {\"role\": \"assistant\", \"content\": [{\"type\": \"text\", \"text\": answer}]},\n                ],\n            }\n\n    if source_id == \"lmms-lab/DocVQA\":\n        query = record.get(\"query\")\n        question = _pick_first_text(\n            record.get(\"question\"),\n            query.get(\"en\") if isinstance(query, dict) else query,\n            query.get(\"question\") if isinstance(query, dict) else None,\n        )\n        answer = _pick_majority_answer(record.get(\"answers\") or record.get(\"answer\"))\n        image_value = record.get(\"image\") or record.get(\"page_image\")\n        if question and answer and image_value is not None:\n            image_path = _save_named_image(source_id, row_id, image_value, asset_dir)\n            return {\n                \"id\": row_id,\n                \"source_id\": source_id,\n                \"license\": \"Apache-2.0\",\n                \"thinking\": \"off\",\n                \"chat_template_kwargs\": {\"enable_thinking\": False},\n                \"category\": \"utility_tasks\",\n                \"modality\": \"image_text\",\n                \"messages\": [\n                    {\n                        \"role\": \"user\",\n                        \"content\": [\n                            make_image_block(image_path),\n                            {\"type\": \"text\", \"text\": question},\n                        ],\n                    },\n                    {\"role\": \"assistant\", \"content\": [{\"type\": \"text\", \"text\": answer}]},\n                ],\n            }\n\n    if source_id == \"HuggingFaceM4/Docmatix\":\n        image_value = record.get(\"image\")\n        if image_value is None and isinstance(record.get(\"images\"), list) and record[\"images\"]:\n            image_value = record[\"images\"][0]\n        qa_pairs = record.get(\"texts\")\n        if isinstance(qa_pairs, list):\n            for index, pair in enumerate(qa_pairs):\n                if not isinstance(pair, dict):\n                    continue\n                question = _pick_first_text(pair.get(\"user\"), pair.get(\"question\"), pair.get(\"prompt\"))\n                answer = _pick_first_text(pair.get(\"assistant\"), pair.get(\"answer\"), pair.get(\"response\"))\n                if question and answer and image_value is not None:\n                    image_path = _save_named_image(source_id, f\"{row_id}-{index}\", image_value, asset_dir)\n                    return {\n                        \"id\": f\"{row_id}-{index}\",\n                        \"source_id\": source_id,\n                        \"license\": \"MIT\",\n                        \"thinking\": \"off\",\n                        \"chat_template_kwargs\": {\"enable_thinking\": False},\n                        \"category\": \"utility_tasks\",\n                        \"modality\": \"image_text\",\n                        \"messages\": [\n                            {\n                                \"role\": \"user\",\n                                \"content\": [\n                                    make_image_block(image_path),\n                                    {\"type\": \"text\", \"text\": question},\n                                ],\n                            },\n                            {\"role\": \"assistant\", \"content\": [{\"type\": \"text\", \"text\": answer}]},\n                        ],\n                    }\n    return None\n\n\ndef render_prompt_messages_to_text(prompt_messages: Any) -> str:\n    if not isinstance(prompt_messages, list):\n        return \"\"\n    parts: list[str] = []\n    for message in prompt_messages:\n        if not isinstance(message, dict) or message.get(\"role\") != \"user\":\n            continue\n        content = message.get(\"content\")\n        if isinstance(content, str):\n            text = normalize_text(content)\n            if text:\n                parts.append(text)\n        elif isinstance(content, list):\n            for block in content:\n                if isinstance(block, dict) and block.get(\"type\") == \"text\":\n                    text = normalize_text(block.get(\"text\"))\n                    if text:\n                        parts.append(text)\n    return \" \".join(parts).strip()\n\n\ndef normalize_messages_for_storage(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    return normalize_messages_as_blocks(messages)\n\n\ndef build_text_row_from_record(\n    source_id: str,\n    record: dict[str, Any],\n    *,\n    row_id: str,\n    license_name: str,\n    category: str,\n) -> dict[str, Any] | None:\n    messages = record.get(\"messages\")\n    if isinstance(messages, list) and messages:\n        normalized = normalize_messages_for_storage(messages)\n    else:\n        prompt = _pick_first_text(\n            record.get(\"prompt\"),\n            record.get(\"instruction\"),\n            record.get(\"question\"),\n            record.get(\"input\"),\n        )\n        answer = _pick_first_text(\n            record.get(\"response\"),\n            record.get(\"output\"),\n            record.get(\"answer\"),\n            record.get(\"chosen\"),\n        )\n        if not prompt or not answer:\n            return None\n        normalized = [\n            {\"role\": \"user\", \"content\": [{\"type\": \"text\", \"text\": prompt}]},\n            {\"role\": \"assistant\", \"content\": [{\"type\": \"text\", \"text\": answer}]},\n        ]\n    return {\n        \"id\": row_id,\n        \"source_id\": source_id,\n        \"license\": license_name,\n        \"thinking\": \"off\",\n        \"chat_template_kwargs\": {\"enable_thinking\": False},\n        \"category\": category,\n        \"modality\": \"text\",\n        \"messages\": normalized,\n    }\n\n\ndef extract_preference_triplet(record: dict[str, Any]) -> tuple[str, str, str] | None:\n    prompt = _pick_first_text(\n        record.get(\"prompt\"),\n        record.get(\"instruction\"),\n        record.get(\"question\"),\n        record.get(\"input\"),\n    )\n    chosen = _pick_first_text(record.get(\"chosen\"))\n    rejected = _pick_first_text(record.get(\"rejected\"))\n    if prompt and chosen and rejected:\n        return prompt, chosen, rejected\n    a = _pick_first_text(record.get(\"response_a\"), record.get(\"answer_a\"), record.get(\"candidate_a\"))\n    b = _pick_first_text(record.get(\"response_b\"), record.get(\"answer_b\"), record.get(\"candidate_b\"))\n    winner = _pick_first_text(record.get(\"winner\"), record.get(\"label\"), record.get(\"preference\")).upper()\n    if prompt and a and b and winner:\n        if winner in {\"A\", \"LEFT\", \"1\", \"CHOICE_A\"}:\n            return prompt, a, b\n        if winner in {\"B\", \"RIGHT\", \"2\", \"CHOICE_B\"}:\n            return prompt, b, a\n    return None\n\n\ndef approximate_row_text(row: dict[str, Any]) -> str:\n    parts: list[str] = []\n    for message in row.get(\"messages\", []):\n        if not isinstance(message, dict):\n            continue\n        content = message.get(\"content\")\n        if isinstance(content, str):\n            text = normalize_text(content)\n            if text:\n                parts.append(text)\n        elif isinstance(content, list):\n            for block in content:\n                if isinstance(block, dict) and block.get(\"type\") == \"text\":\n                    text = normalize_text(block.get(\"text\"))\n                    if text:\n                        parts.append(text)\n                elif isinstance(block, dict) and block.get(\"type\") == \"image\":\n                    parts.append(\"[image]\")\n    return \"\\n\".join(parts).strip()\n\n\ndef _requirement_name(line: str) -> str:\n    stripped = line.strip()\n    if not stripped or stripped.startswith(\"#\"):\n        return \"\"\n    for token in (\"[\", \">\", \"=\", \"<\", \"!\", \"~\"):\n        if token in stripped:\n            stripped = stripped.split(token, 1)[0]\n            break\n    return stripped.strip().lower()\n\n\ndef select_supplemental_requirements(requirements: Iterable[str]) -> list[str]:\n    selected: list[str] = []\n    seen: set[str] = set()\n    for line in requirements:\n        requirement = line.strip()\n        name = _requirement_name(requirement)\n        if not requirement or not name or name in CORE_STACK_PACKAGES or name in seen:\n            continue\n        selected.append(requirement)\n        seen.add(name)\n    return selected\n\n\ndef _normalize_cuda_tag(cuda_version: str) -> str:\n    normalized = str(cuda_version).strip()\n    if not normalized:\n        raise ValueError(\"cuda_version must be non-empty\")\n    if normalized not in {\"11.8\", \"12.1\", \"12.4\", \"12.6\", \"12.8\", \"13.0\"}:\n        raise ValueError(f\"unsupported CUDA version for Unsloth matrix install: {normalized}\")\n    return f\"cu{normalized.replace('.', '')}\"\n\n\ndef resolve_unsloth_matrix_install_command(torch_version: str, cuda_version: str) -> str:\n    version_text = str(torch_version).split(\"+\", 1)[0].strip()\n    version = Version(version_text)\n    if version <= Version(\"2.1.0\"):\n        raise ValueError(f\"unsupported torch version for Unsloth matrix install: {torch_version}\")\n    if version <= Version(\"2.1.1\"):\n        torch_tag = \"torch211\"\n    elif version <= Version(\"2.1.2\"):\n        torch_tag = \"torch212\"\n    elif version < Version(\"2.3.0\"):\n        torch_tag = \"torch220\"\n    elif version < Version(\"2.4.0\"):\n        torch_tag = \"torch230\"\n    elif version < Version(\"2.5.0\"):\n        torch_tag = \"torch240\"\n    elif version < Version(\"2.5.1\"):\n        torch_tag = \"torch250\"\n    elif version <= Version(\"2.5.1\"):\n        torch_tag = \"torch251\"\n    elif version < Version(\"2.7.0\"):\n        torch_tag = \"torch260\"\n    elif version < Version(\"2.7.9\"):\n        torch_tag = \"torch270\"\n    elif version < Version(\"2.8.0\"):\n        torch_tag = \"torch271\"\n    elif version < Version(\"2.8.9\"):\n        torch_tag = \"torch280\"\n    elif version < Version(\"2.9.1\"):\n        torch_tag = \"torch290\"\n    elif version < Version(\"2.9.2\"):\n        torch_tag = \"torch291\"\n    else:\n        raise ValueError(f\"unsupported torch version for Unsloth matrix install: {torch_version}\")\n\n    cuda_tag = _normalize_cuda_tag(cuda_version)\n    matrix_tag = f\"{cuda_tag}-{torch_tag}\"\n    return (\n        \"pip install --upgrade pip && \"\n        \"pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git#egg=unsloth_zoo && \"\n        f'pip install \"unsloth[{matrix_tag}] @ git+https://github.com/unslothai/unsloth.git\" '\n        \"--no-build-isolation\"\n    )\n\n\ndef materialize_processor_ready_sft_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    prepared: list[dict[str, Any]] = []\n    for row in rows:\n        messages = normalize_messages_as_blocks(list(row.get(\"messages\") or []))\n        images: list[Image.Image] = []\n        normalized_messages: list[dict[str, Any]] = []\n        for message in messages:\n            new_content: list[dict[str, Any]] = []\n            for block in message.get(\"content\", []):\n                if not isinstance(block, dict):\n                    continue\n                if block.get(\"type\") == \"text\":\n                    text = normalize_text(block.get(\"text\"))\n                    if text:\n                        new_content.append({\"type\": \"text\", \"text\": text})\n                elif block.get(\"type\") == \"image\":\n                    image_path = block.get(\"image_path\") or block.get(\"path\") or block.get(\"image\")\n                    if not isinstance(image_path, str) or not image_path.strip():\n                        raise FileNotFoundError(f\"missing concrete image path for row {row.get('id')}\")\n                    resolved = Path(image_path).expanduser().resolve()\n                    if not resolved.exists():\n                        raise FileNotFoundError(f\"missing image file for row {row.get('id')}: {resolved}\")\n                    with Image.open(resolved) as image:\n                        images.append(image.convert(\"RGB\").copy())\n                    new_content.append({\"type\": \"image\"})\n            normalized_messages.append({\"role\": message.get(\"role\", \"user\"), \"content\": new_content})\n        item: dict[str, Any] = {\n            \"id\": str(row.get(\"id\") or f\"row-{len(prepared):08d}\"),\n            \"messages\": normalized_messages,\n        }\n        if images:\n            item[\"images\"] = images\n            item[\"image\"] = images[0]\n        prepared.append(item)\n    return prepared\n\n\ndef resolve_dpo_policy(\n    *,\n    is_multimodal_run: bool,\n    preference_has_images: bool,\n    experimental_dpo_enabled: bool,\n) -> dict[str, Any]:\n    if is_multimodal_run and not preference_has_images and not experimental_dpo_enabled:\n        return {\n            \"status\": \"skipped\",\n            \"reason\": \"skipped_text_only_preferences_on_multimodal_run\",\n            \"run_dpo\": False,\n        }\n    if experimental_dpo_enabled:\n        return {\n            \"status\": \"experimental\",\n            \"reason\": \"experimental_dpo_enabled\",\n            \"run_dpo\": True,\n        }\n    return {\n        \"status\": \"enabled\",\n        \"reason\": \"preferences_support_requested_path\",\n        \"run_dpo\": True,\n    }\n\n\ndef choose_final_download_target(\n    *,\n    final_export_files: list[Path],\n    zip_bundle_path: Path,\n    single_file_size_limit_bytes: int,\n) -> dict[str, Any]:\n    if len(final_export_files) != 1:\n        return {\"download_mode\": \"zip_bundle\", \"download_path\": zip_bundle_path}\n    file_path = final_export_files[0]\n    if file_path.stat().st_size > single_file_size_limit_bytes:\n        return {\"download_mode\": \"zip_bundle\", \"download_path\": zip_bundle_path}\n    return {\"download_mode\": \"single_file\", \"download_path\": file_path}\n\n\ndef create_final_report_payload(\n    *,\n    what_changed: list[str],\n    what_was_verified: list[str],\n    what_remains_unproven: list[str],\n    highest_risk_unresolved_issue: str,\n    exact_next_step: str,\n) -> dict[str, Any]:\n    return {\n        \"what_changed\": what_changed,\n        \"what_was_verified\": what_was_verified,\n        \"what_remains_unproven\": what_remains_unproven,\n        \"highest_risk_unresolved_issue\": highest_risk_unresolved_issue,\n        \"exact_next_step\": exact_next_step,\n    }\n\n\ndef collect_file_checksums(root: str | Path) -> dict[str, str]:\n    base = Path(root).expanduser().resolve()\n    checksums: dict[str, str] = {}\n    if not base.exists():\n        return checksums\n    for path in sorted(base.rglob(\"*\")):\n        if path.is_file():\n            checksums[str(path.relative_to(base))] = sha256_file(path)\n    return checksums\n\n\ndef build_zip_bundle(zip_path: str | Path, files: Iterable[str | Path]) -> Path:\n    destination = Path(zip_path).expanduser().resolve()\n    _ensure_parent(destination)\n    with zipfile.ZipFile(destination, \"w\", compression=zipfile.ZIP_DEFLATED) as bundle:\n        for item in files:\n            target = Path(item).expanduser().resolve()\n            if target.is_file():\n                bundle.write(target, arcname=target.name)\n    return destination\n"
EMBEDDED_REQUIREMENTS_TEXT = "unsloth>=2026.2.0\nunsloth_zoo>=2026.2.0\ntorch>=2.5.0\ntransformers>=5.0.0,<6.0.0\ntrl>=0.25.0\ndatasets>=3.3.0\naccelerate>=1.5.0\npeft>=0.17.0\nbitsandbytes>=0.47.0\nhuggingface-hub>=1.4.0\nsentencepiece>=0.2.0\nsafetensors>=0.6.0\ntokenizers>=0.22.0\nnumpy>=2.2.0\npandas>=2.3.0\npyyaml>=6.0.2\ntqdm>=4.67.0\nPillow>=10.4.0\ntorchvision>=0.20.0\njsonschema>=4.23.0\nlangdetect>=1.0.9\nnbformat>=5.10.4\nipykernel>=6.30.1\n"
OUTPUT_PATH = REPO_ROOT / "notebooks" / "91_colab_unified_unsloth_first.ipynb"

for path in [
    WORK_ROOT,
    WORKSPACE_ROOT,
    REPORTS_ROOT / "bootstrap",
    FINAL_ROOT,
    RUNS_ROOT,
    ARTIFACTS_ROOT,
    CHECKSUMS_ROOT,
    *RUN_STAGE_DIRS.values(),
]:
    path.mkdir(parents=True, exist_ok=True)

os.chdir(WORK_ROOT)
print(json.dumps({
    "run_id": RUN_ID,
    "work_root": str(WORK_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "run_root": str(RUN_ROOT),
    "install_strategy": INSTALL_STRATEGY,
    "profile": PROFILE,
    "base_model": BASE_MODEL,
    "identity_repo_id": IDENTITY_REPO_ID,
}, indent=2))


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore

def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

def mount_drive_safely(mountpoint: str = "/content/drive") -> dict[str, object]:
    mount_path = Path(mountpoint)
    report = {
        "mountpoint": mountpoint,
        "exists_before": mount_path.exists(),
        "is_mount_before": os.path.ismount(mountpoint),
        "entries_before": [],
        "recovery_action": None,
        "mounted": False,
    }
    if mount_path.exists() and mount_path.is_dir():
        report["entries_before"] = sorted(item.name for item in mount_path.iterdir())[:20]
    if not IN_COLAB:
        report["mounted"] = False
        report["recovery_action"] = "not_running_in_colab"
        return report
    if os.path.ismount(mountpoint):
        report["mounted"] = True
        report["recovery_action"] = "already_mounted"
        return report
    if mount_path.exists():
        if mount_path.is_file() or mount_path.is_symlink():
            mount_path.unlink()
            report["recovery_action"] = "removed_file_or_symlink"
        elif mount_path.is_dir() and any(mount_path.iterdir()):
            shutil.rmtree(mount_path, ignore_errors=True)
            report["recovery_action"] = "removed_nonempty_directory"
    drive.mount(mountpoint, force_remount=bool(report["recovery_action"]))
    report["mounted"] = os.path.ismount(mountpoint)
    return report

DRIVE_MOUNT_RESULT = mount_drive_safely("/content/drive")
write_json(REPORTS_ROOT / "bootstrap" / "drive_mount.json", DRIVE_MOUNT_RESULT)
write_json(BOOTSTRAP_ROOT / "drive_mount.json", DRIVE_MOUNT_RESULT)
print(json.dumps(DRIVE_MOUNT_RESULT, indent=2))


In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata
import json
import os
import re
import subprocess
import sys
from pathlib import Path

CORE_UNSLOTH_MANAGED = {
    "torch",
    "transformers",
    "trl",
    "accelerate",
    "peft",
    "bitsandbytes",
    "torchvision",
    "unsloth",
    "unsloth_zoo",
}

def _requirement_name(line: str) -> str:
    candidate = line.strip()
    if not candidate or candidate.startswith("#"):
        return ""
    for token in ("[", ">", "=", "<", "!", "~"):
        if token in candidate:
            candidate = candidate.split(token, 1)[0]
            break
    return candidate.strip().lower()

def select_supplemental_requirements(requirements_text: str) -> list[str]:
    selected = []
    seen = set()
    for raw in requirements_text.splitlines():
        name = _requirement_name(raw)
        line = raw.strip()
        if not line or not name or name in CORE_UNSLOTH_MANAGED or name in seen:
            continue
        selected.append(line)
        seen.add(name)
    return selected

def package_versions(packages: list[str]) -> dict[str, str | None]:
    report = {}
    for package in packages:
        try:
            report[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            report[package] = None
    return report

def detect_torch_cuda() -> tuple[str, str]:
    try:
        import torch
        cuda_version = getattr(torch.version, "cuda", None) or os.environ.get("CUDA_VERSION") or "12.4"
        return str(torch.__version__), str(cuda_version)
    except Exception:
        return "2.5.0", os.environ.get("CUDA_VERSION", "12.4")

def build_unsloth_matrix_install_command(torch_version: str, cuda_version: str) -> str:
    match = re.match(r"^(\d+)\.(\d+)", str(torch_version))
    if not match:
        raise ValueError(f"unsupported torch version: {torch_version}")
    major, minor = int(match.group(1)), int(match.group(2))
    if (major, minor) >= (2, 9):
        torch_tag = "torch290"
    elif (major, minor) >= (2, 8):
        torch_tag = "torch280"
    elif (major, minor) >= (2, 7):
        torch_tag = "torch270"
    elif (major, minor) >= (2, 6):
        torch_tag = "torch260"
    elif (major, minor) >= (2, 5):
        torch_tag = "torch250"
    elif (major, minor) >= (2, 4):
        torch_tag = "torch240"
    else:
        raise ValueError(f"unsupported torch version: {torch_version}")
    cuda_text = str(cuda_version).strip().replace(".", "")
    matrix_tag = f"cu{cuda_text}-{torch_tag}"
    return (
        "pip install --upgrade pip && "
        "pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git#egg=unsloth_zoo && "
        f"pip install \"unsloth[{matrix_tag}] @ git+https://github.com/unslothai/unsloth.git\" "
        "--no-build-isolation"
    )

def pip_install(args: list[str]) -> None:
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)

SUPPLEMENTAL_REQUIREMENTS = select_supplemental_requirements(EMBEDDED_REQUIREMENTS_TEXT)
install_report = {
    "install_strategy": INSTALL_STRATEGY,
    "supplemental_requirements": SUPPLEMENTAL_REQUIREMENTS,
    "primary_command": [sys.executable, "-m", "pip", "install", "unsloth"],
    "fallback_command": None,
    "used_fallback": False,
}

pip_install(["install", "--upgrade", "pip"])
try:
    pip_install(["install", "unsloth"])
    importlib.import_module("unsloth")
except Exception:
    torch_version, cuda_version = detect_torch_cuda()
    fallback_command = build_unsloth_matrix_install_command(torch_version, cuda_version)
    install_report["fallback_command"] = fallback_command
    install_report["used_fallback"] = True
    subprocess.run(fallback_command, shell=True, check=True)
    importlib.import_module("unsloth")

if SUPPLEMENTAL_REQUIREMENTS:
    pip_install(["install", *SUPPLEMENTAL_REQUIREMENTS])

install_report["versions"] = package_versions([
    "unsloth",
    "unsloth_zoo",
    "torch",
    "transformers",
    "trl",
    "accelerate",
    "peft",
    "bitsandbytes",
    "torchvision",
    "datasets",
    "huggingface-hub",
    "pillow",
])
write_json(REPORTS_ROOT / "bootstrap" / "install_strategy_and_versions.json", install_report)
write_json(BOOTSTRAP_ROOT / "install_strategy_and_versions.json", install_report)
print(json.dumps(install_report, indent=2))


In [ ]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

EMBEDDED_RUNTIME_DIR = WORK_ROOT / "_embedded_runtime"
EMBEDDED_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_RUNTIME_PATH = EMBEDDED_RUNTIME_DIR / "colab_unified_unsloth_first.py"
EMBEDDED_RUNTIME_PATH.write_text(EMBEDDED_RUNTIME_SOURCE, encoding="utf-8")
if str(EMBEDDED_RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(EMBEDDED_RUNTIME_DIR))

EMBEDDED_CONFIG_DIR = WORK_ROOT / "_embedded_configs"
EMBEDDED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
for name, text in EMBEDDED_CONFIG_TEXT.items():
    (EMBEDDED_CONFIG_DIR / name).write_text(text, encoding="utf-8")

from colab_unified_unsloth_first import (
    IDENTITY_ALLOW_PATTERNS,
    build_multimodal_row_from_record,
    build_text_row_from_record,
    build_zip_bundle,
    choose_final_download_target,
    collect_file_checksums,
    create_final_report_payload,
    extract_preference_triplet,
    materialize_identity_placeholder_assets,
    materialize_processor_ready_sft_rows,
    resolve_dpo_policy,
    resolve_unsloth_matrix_install_command,
    sha256_file,
)

import yaml
from datasets import load_dataset
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoProcessor
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

DATASET_MIXTURE = yaml.safe_load(EMBEDDED_CONFIG_TEXT["dataset_mixture.yaml"])
DATASET_ALLOWLIST = yaml.safe_load(EMBEDDED_CONFIG_TEXT["dataset_sources_allowlist.yaml"])
TRAIN_SFT_CFG = yaml.safe_load(EMBEDDED_CONFIG_TEXT["train_sft.yaml"])
TRAIN_DPO_CFG = yaml.safe_load(EMBEDDED_CONFIG_TEXT["train_dpo.yaml"])
RUN_PROFILES = yaml.safe_load(EMBEDDED_CONFIG_TEXT["run_profiles.yaml"])
PROFILE_CFG = RUN_PROFILES["profiles"][PROFILE]

import_report = {
    "runtime_module": str(EMBEDDED_RUNTIME_PATH),
    "identity_allow_patterns": IDENTITY_ALLOW_PATTERNS,
    "required_imports": ["FastVisionModel", "UnslothVisionDataCollator", "AutoProcessor", "load_dataset", "snapshot_download"],
    "profile": PROFILE,
    "profile_cfg": PROFILE_CFG,
}
write_json(BOOTSTRAP_ROOT / "embedded_runtime_imports.json", import_report)
print(json.dumps(import_report, indent=2))


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                yield json.loads(line)

IDENTITY_STAGE_DIR = DATASET_RUN_ROOT / "identity"
IDENTITY_STAGE_DIR.mkdir(parents=True, exist_ok=True)
identity_snapshot_dir = snapshot_download(
    repo_id=IDENTITY_REPO_ID,
    repo_type="dataset",
    local_dir=str(IDENTITY_STAGE_DIR),
    allow_patterns=IDENTITY_ALLOW_PATTERNS,
)
sft_identity_path = IDENTITY_STAGE_DIR / "sft_dataset.jsonl"
preference_identity_path = IDENTITY_STAGE_DIR / "preference_dataset.jsonl"
if not sft_identity_path.exists() or not preference_identity_path.exists():
    raise FileNotFoundError("identity bootstrap failed to produce canonical filenames")
identity_report = {
    "repo_id": IDENTITY_REPO_ID,
    "snapshot_dir": str(identity_snapshot_dir),
    "sft_path": str(sft_identity_path),
    "preference_path": str(preference_identity_path),
    "sft_rows": sum(1 for _ in iter_jsonl(sft_identity_path)),
    "preference_rows": sum(1 for _ in iter_jsonl(preference_identity_path)),
}
write_json(REPORTS_ROOT / "bootstrap" / "identity_download.json", identity_report)
write_json(BOOTSTRAP_ROOT / "identity_download.json", identity_report)
print(json.dumps(identity_report, indent=2))


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

OPEN_SFT_PATH = FINAL_ROOT / "full_sft.jsonl"
OPEN_PREFERENCE_PATH = FINAL_ROOT / "full_preferences.jsonl"
OPEN_ASSET_ROOT = DATASET_RUN_ROOT / "open_assets"
OPEN_ASSET_ROOT.mkdir(parents=True, exist_ok=True)

def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

allowlist_map = {
    item["source_id"]: item
    for item in DATASET_ALLOWLIST.get("sources", [])
    if isinstance(item, dict) and item.get("enabled")
}
open_sft_rows = []
open_preference_rows = []
source_results = []

for source_cfg in DATASET_MIXTURE.get("sources", []):
    source_id = source_cfg.get("source_id")
    if source_id not in allowlist_map:
        continue
    split_name = allowlist_map[source_id].get("default_split", "train")
    try:
        dataset = load_dataset(source_id, split=split_name, streaming=True)
        scanned = 0
        accepted = 0
        if source_cfg.get("modality") == "image_text":
            for record in dataset:
                scanned += 1
                row = build_multimodal_row_from_record(
                    source_id,
                    record,
                    asset_root=OPEN_ASSET_ROOT,
                    row_id=f"{source_id.replace('/', '__')}-{scanned:08d}",
                )
                if row is None:
                    if scanned >= 64:
                        break
                    continue
                open_sft_rows.append(row)
                accepted += 1
                if accepted >= 8:
                    break
        else:
            for record in dataset:
                scanned += 1
                row = build_text_row_from_record(
                    source_id,
                    record,
                    row_id=f"{source_id.replace('/', '__')}-{scanned:08d}",
                    license_name=allowlist_map[source_id].get("license", "unknown"),
                    category=(allowlist_map[source_id].get("category") or ["utility_tasks"])[0],
                )
                if row is not None:
                    open_sft_rows.append(row)
                    accepted += 1
                triplet = extract_preference_triplet(record)
                if triplet is not None and source_id in set(DATASET_MIXTURE.get("preferences", {}).get("enabled_sources", [])):
                    prompt, chosen, rejected = triplet
                    open_preference_rows.append({
                        "id": f"{source_id.replace('/', '__')}-pref-{accepted:08d}",
                        "source_id": source_id,
                        "prompt": prompt,
                        "chosen": chosen,
                        "rejected": rejected,
                    })
                if accepted >= 8 and scanned >= 32:
                    break
        source_results.append({"source_id": source_id, "accepted": accepted, "scanned": scanned, "status": "ok"})
    except Exception as exc:
        source_results.append({"source_id": source_id, "accepted": 0, "scanned": 0, "status": "failed", "error": str(exc)})

identity_sft_rows = load_jsonl(sft_identity_path)
identity_preference_rows = load_jsonl(preference_identity_path)
materialized_identity_rows, identity_materialization = materialize_identity_placeholder_assets(
    identity_sft_rows,
    DATASET_RUN_ROOT / "identity_assets",
)
merged_sft_rows = materialized_identity_rows + open_sft_rows
merged_preference_rows = identity_preference_rows + open_preference_rows
write_jsonl(OPEN_SFT_PATH, merged_sft_rows)
write_jsonl(OPEN_PREFERENCE_PATH, merged_preference_rows)

dataset_report = {
    "final_sft_path": str(OPEN_SFT_PATH),
    "final_preferences_path": str(OPEN_PREFERENCE_PATH),
    "identity_sft_rows": len(identity_sft_rows),
    "identity_preference_rows": len(identity_preference_rows),
    "open_sft_rows": len(open_sft_rows),
    "open_preference_rows": len(open_preference_rows),
    "merged_sft_rows": len(merged_sft_rows),
    "merged_preference_rows": len(merged_preference_rows),
    "identity_materialization": identity_materialization,
    "source_results": source_results,
}
write_json(DATASET_RUN_ROOT / "dataset_build_report.json", dataset_report)
print(json.dumps(dataset_report, indent=2))


In [ ]:
from __future__ import annotations

import json

with OPEN_SFT_PATH.open("r", encoding="utf-8") as handle:
    merged_sft_rows = [json.loads(line) for line in handle if line.strip()]

processor_ready_rows = materialize_processor_ready_sft_rows(merged_sft_rows)
multimodal_rows = [row for row in processor_ready_rows if row.get("images")]
if not multimodal_rows:
    raise RuntimeError("blocking: merged SFT dataset has zero concrete image rows after materialization")

processor_ready_report = {
    "processor_ready_rows": len(processor_ready_rows),
    "multimodal_rows": len(multimodal_rows),
    "sample_ids": [row["id"] for row in multimodal_rows[:5]],
}
write_json(DATASET_RUN_ROOT / "processor_ready_report.json", processor_ready_report)
print(json.dumps(processor_ready_report, indent=2))


In [ ]:
from __future__ import annotations

import json
import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

SFT_MODEL_DIR = SFT_RUN_ROOT / "artifacts" / "sft_model"
SFT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    load_in_4bit=True,
    trust_remote_code=True,
)
train_dataset = Dataset.from_list(multimodal_rows)
trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir=str(SFT_MODEL_DIR),
        per_device_train_batch_size=PROFILE_CFG["sft"]["per_device_train_batch_size"],
        gradient_accumulation_steps=PROFILE_CFG["sft"]["gradient_accumulation_steps"],
        max_steps=min(int(TRAIN_SFT_CFG["training"]["max_steps"]), 50),
        logging_steps=int(TRAIN_SFT_CFG["training"]["logging_steps"]),
        save_steps=int(TRAIN_SFT_CFG["training"]["save_steps"]),
        bf16=True,
        max_length=None,
    ),
)
trainer.train()
processor.save_pretrained(str(SFT_MODEL_DIR / "processor"))
trainer.model.save_pretrained(str(SFT_MODEL_DIR))
sft_report = {
    "output_dir": str(SFT_MODEL_DIR),
    "rows_used": len(multimodal_rows),
    "base_model": BASE_MODEL,
    "max_length": None,
    "run_training_default": True,
}
write_json(SFT_RUN_ROOT / "sft_report.json", sft_report)
print(json.dumps(sft_report, indent=2))


In [ ]:
from __future__ import annotations

import json

EXPORT_SFT_DIR = EXPORT_SFT_RUN_ROOT / "artifacts"
EXPORT_SFT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR = EXPORT_SFT_DIR / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)
FINAL_EXPORT_FILES = []

if hasattr(model, "save_pretrained_gguf"):
    model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method=EXPORT_QUANTIZATION_METHODS)

for path in sorted(GGUF_DIR.rglob("*")):
    if path.is_file():
        FINAL_EXPORT_FILES.append(path)

if not FINAL_EXPORT_FILES:
    raise RuntimeError("blocking: GGUF export produced no files")

ZIP_BUNDLE_PATH = EXPORT_SFT_DIR / "final_deliverables.zip"
build_zip_bundle(ZIP_BUNDLE_PATH, FINAL_EXPORT_FILES)

export_report = {
    "export_dir": str(EXPORT_SFT_DIR),
    "gguf_dir": str(GGUF_DIR),
    "gguf_files": [str(path) for path in FINAL_EXPORT_FILES],
    "zip_bundle_path": str(ZIP_BUNDLE_PATH),
    "export_quantization_methods": EXPORT_QUANTIZATION_METHODS,
}
write_json(EXPORT_SFT_RUN_ROOT / "gguf_export_report.json", export_report)
print(json.dumps(export_report, indent=2))


In [ ]:
from __future__ import annotations

import json

preference_has_images = False
dpo_policy = resolve_dpo_policy(
    is_multimodal_run=True,
    preference_has_images=preference_has_images,
    experimental_dpo_enabled=EXPERIMENTAL_DPO,
)
write_json(DPO_RUN_ROOT / "dpo_policy.json", dpo_policy)
if dpo_policy["run_dpo"]:
    dpo_result = {"status": "not_implemented_in_default_path", "reason": "experimental_only"}
else:
    dpo_result = dpo_policy
write_json(DPO_RUN_ROOT / "dpo_result.json", dpo_result)
print(json.dumps(dpo_result, indent=2))


In [ ]:
from __future__ import annotations

import json

final_export_root = EXPORT_FINAL_RUN_ROOT / "artifacts"
final_export_root.mkdir(parents=True, exist_ok=True)
selected_artifact_root = EXPORT_SFT_DIR
selected_export_files = FINAL_EXPORT_FILES
selected_zip_path = ZIP_BUNDLE_PATH
if dpo_policy["run_dpo"] and (DPO_RUN_ROOT / "final_deliverables.zip").exists():
    selected_artifact_root = DPO_RUN_ROOT
    selected_export_files = list((DPO_RUN_ROOT / "artifacts").rglob("*.gguf"))
    selected_zip_path = DPO_RUN_ROOT / "final_deliverables.zip"

eval_report = {
    "selected_artifact_root": str(selected_artifact_root),
    "selected_export_files": [str(path) for path in selected_export_files],
    "selected_zip_path": str(selected_zip_path),
    "selection_order": "post-DPO export bundle if fully successful, otherwise SFT export bundle",
}
write_json(EVAL_RUN_ROOT / "eval_selection_report.json", eval_report)
print(json.dumps(eval_report, indent=2))


In [ ]:
from __future__ import annotations

import json
import shutil
import sys

if not selected_export_files and not Path(selected_zip_path).exists():
    raise RuntimeError("blocking: no final export artifact exists to package or download")

RUN_ARTIFACTS_DIR = RUN_ROOT / "artifacts"
RUN_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
mirrored_files = []
for artifact_path in [Path(path) for path in selected_export_files]:
    if artifact_path.exists():
        destination = RUN_ARTIFACTS_DIR / artifact_path.name
        shutil.copy2(artifact_path, destination)
        mirrored_files.append(destination)
if Path(selected_zip_path).exists():
    zip_destination = RUN_ARTIFACTS_DIR / Path(selected_zip_path).name
    shutil.copy2(selected_zip_path, zip_destination)
else:
    zip_destination = RUN_ARTIFACTS_DIR / "final_deliverables.zip"

final_target = choose_final_download_target(
    final_export_files=mirrored_files,
    zip_bundle_path=zip_destination,
    single_file_size_limit_bytes=1900 * 1024 * 1024,
)
final_report = create_final_report_payload(
    what_changed=[
        "Created notebook 91 as the canonical Colab-first surface.",
        "Made Unsloth-first bootstrap the default install strategy.",
        "Embedded runtime helper code and config snapshots into the notebook.",
        "Added HF identity auto-download, safe Drive mount recovery, and automatic final download.",
    ],
    what_was_verified=[
        "Notebook source compiles cell-by-cell during generation.",
        "Helper-layer tests cover install filtering, DPO policy, processor-ready multimodal rows, and final bundle selection.",
        "The notebook records bootstrap reports for drive mount, identity download, and install strategy.",
    ],
    what_remains_unproven=[
        "A real Colab G4 execution for the full multimodal training and GGUF export path.",
        "Whether the current text-only preference surface supports a stable multimodal DPO stage.",
    ],
    highest_risk_unresolved_issue="The first proof-bearing Colab G4 run still needs to confirm that upstream multimodal dataset schemas and surrogate identity images behave acceptably end to end.",
    exact_next_step="Run notebooks/91_colab_unified_unsloth_first.ipynb on a real Colab G4 runtime and inspect the emitted workspace/runs tree for the generated run ID.",
)
final_checksums = collect_file_checksums(RUN_ARTIFACTS_DIR)
write_json(RUN_ROOT / "checksums" / "artifacts.json", final_checksums)
write_json(REPORTS_ROOT / "final_run_report.json", final_report)
(REPORTS_ROOT / "final_run_report.md").write_text(
    "\n".join([
        "# Final Run Report",
        "",
        "## what_changed",
        *[f"- {item}" for item in final_report["what_changed"]],
        "",
        "## what_was_verified",
        *[f"- {item}" for item in final_report["what_was_verified"]],
        "",
        "## what_remains_unproven",
        *[f"- {item}" for item in final_report["what_remains_unproven"]],
        "",
        "## highest_risk_unresolved_issue",
        final_report["highest_risk_unresolved_issue"],
        "",
        "## exact_next_step",
        final_report["exact_next_step"],
        "",
    ]) + "\n",
    encoding="utf-8",
)
STATUS_PATH.write_text(
    json.dumps({"run_id": RUN_ID, "final_target": str(final_target["download_path"]), "download_mode": final_target["download_mode"]}, indent=2) + "\n",
    encoding="utf-8",
)
SUMMARY_PATH.write_text(
    "# Notebook 91 Summary\n\n"
    f"- run_id: {RUN_ID}\n"
    f"- install_strategy: {INSTALL_STRATEGY}\n"
    f"- final_target: {final_target['download_path']}\n"
    f"- download_mode: {final_target['download_mode']}\n",
    encoding="utf-8",
)
if "google.colab" in sys.modules:
    from google.colab import files  # type: ignore
    files.download(str(final_target["download_path"]))
print(json.dumps({"final_target": str(final_target["download_path"]), "download_mode": final_target["download_mode"]}, indent=2))
